In [ ]:
from src.data.data import *
from src.embedor import *
from src.plotting import *
from sklearn.neighbors import radius_neighbors_graph, kneighbors_graph
import networkx as nx
import os
save_path = '/home/tristan/Research/Sp25/embedor/outputs/noise_model/'
os.makedirs(save_path, exist_ok=True)

%load_ext autoreload

In [ ]:
def signed_distortion(A, B):
    distortions = []
    for i in range(A.shape[0]):
        for j in range(i):
            if A[i, j] != 0:
                distortion = (A[i, j] - B[i, j]) / A[i, j]
                distortions.append(distortion)
    distortions = np.array(distortions)
    return distortions

def noise_model_experiment(
        noiseless_data, 
        clusters, 
        noise_scale, 
        dataset_name, 
        n_iter=30, 
        pmax=0.1, 
        sigma_1_sq = 0.5, 
        sigma_2_sq = 0.09, 
        n_noisy=15,
        epsilon=0.15,
        k=None
    ):
    
    differing_cluster_mask = np.array([clusters[i] != clusters[j] for i in range(len(clusters)) for j in range(len(clusters))]).reshape(len(clusters), len(clusters))
    num_sc_ambient = []
    num_sc_adj = []

    # build noiseless data graph
    if k is None:
        A_noiseless = radius_neighbors_graph(noiseless_data, radius=epsilon, mode='distance', include_self=False)
    else:
        A_noiseless = kneighbors_graph(noiseless_data, n_neighbors=k, mode='distance', include_self=False)
    # for each point, compute the distance to nearest point in other cluster
    adv_distances = []
    for i in range(len(noiseless_data)):
        other_cluster = 1 - clusters[i]
        other_points = noiseless_data[clusters == other_cluster]
        distances = np.linalg.norm(other_points - noiseless_data[i], axis=1)
        adv_distances.append(np.min(distances))
    adv_distances = np.array(adv_distances)
    # convert to probability
    s_i = pmax * np.exp(-adv_distances**2/sigma_1_sq)
    print(f'Maximum s_i: {np.max(s_i)}, required upper bound: {3/(8 * np.exp(1))}')
    assert np.max(s_i) <= 3/(8 * np.exp(1)), "s_i exceeds the required upper bound for the noise model."

    for i in range(n_iter):

        print(f'Iteration {i+1}/{n_iter}')
        # generate noise
        noise = np.random.normal(0, noise_scale, noiseless_data.shape)
        noisy_data = noiseless_data.copy() + noise

        # build noisy data graph
        if k is None:
            A = radius_neighbors_graph(noisy_data, radius=epsilon, mode='distance', include_self=False)
        else:
            A = kneighbors_graph(noisy_data, n_neighbors=k, mode='distance', include_self=False)
        # convert A to networkx graph
        G = nx.from_scipy_sparse_array(A)

        # sample a bernoulli variable for each point based on s_i
        nus = np.array([np.random.binomial(1, s_i[i]) for i in range(len(s_i)) ])
        print(f'Number of points with nus[i] = 1: {np.sum(nus)}')
        A_noiseless_corrupted = A_noiseless.copy()
        for i in range(len(noiseless_data)):
            if nus[i] == 1:
                # sample n_noisy points any other point with nus[j] = 1
                sampleable_indices = np.where(nus == 1)[0]
                sampleable_indices = np.delete(sampleable_indices, np.where(sampleable_indices == i)[0])
                # remove points that are in the same cluster
                sampleable_indices = sampleable_indices[clusters[sampleable_indices] != clusters[i]]
                if len(sampleable_indices) == 0:
                    # if no points are available, skip this point
                    continue
                # sample n_noisy points from sampleable_indices based on distance
                sampleable_points = noiseless_data[sampleable_indices]
                distances = np.linalg.norm(sampleable_points - noiseless_data[i], axis=1)
                # sample n_noisy points from sampleable_points with probability dependent on distance
                prob = np.exp(-distances**2/sigma_2_sq)
                prob = prob / np.sum(prob)
                # sample n_noisy points from sampleable_points with prob
                sampled_points_indices = np.random.choice(len(sampleable_points), size=int(min(n_noisy, len(sampleable_points))), p=prob, replace=True)
                sampled_points = [sampleable_points[index] for index in sampled_points_indices]
                # add sampled points to A_noiseless_corrupted with weight as distance
                for sampled_pt, index, dist in zip(sampled_points, sampleable_indices[sampled_points_indices], distances[sampled_points_indices]):
                    # add edge to A_noiseless_corrupted
                    A_noiseless_corrupted[i, index] = dist
                    A_noiseless_corrupted[index, i] = dist
        
        # convert A_noiseless_corrupted to binary
        A_binarized = np.where(A.toarray() > 0, 1, 0)
        A_noiseless_corrupted_binarized = np.where(A_noiseless_corrupted.toarray() > 0, 1, 0)
        # num shortcut
        num_shortcut = np.sum(A_binarized * differing_cluster_mask)
        num_shortcut_noiseless_corrupted = np.sum(A_noiseless_corrupted_binarized * differing_cluster_mask)
        num_sc_adj.append(num_shortcut_noiseless_corrupted)
        num_sc_ambient.append(num_shortcut)
    
    plt.figure(figsize=(10, 10))
    if noiseless_data.shape[1] != 2:
        noiseless_data = noiseless_data[:, [0,2]]  # Use only the first two dimensions for plotting
    plot_graph_2D(noiseless_data, G, edge_width=0.25, edge_color='black')
    plt.savefig(os.path.join(save_path, f'{dataset_name}_ambient_noise_graph.png'), dpi=300)

    # convert A_noiseless_corrupted to networkx graph
    G_noiseless_corrupted = nx.from_scipy_sparse_array(A_noiseless_corrupted)
    plt.figure(figsize=(10, 10))
    plot_graph_2D(noiseless_data, G_noiseless_corrupted, edge_width=0.25, edge_color='black')
    plt.savefig(os.path.join(save_path, f'{dataset_name}_noiseless_corrupted_graph.png'), dpi=300)

    print(f'Mean, std of number of cluster-bridging edges for ambient noise: {np.mean(num_sc_ambient)}, {np.std(num_sc_ambient)}')
    print(f'Mean, std of number of cluster-bridging edges for adjacency: {np.mean(num_sc_adj)}, {np.std(num_sc_adj)}')
    return G, G_noiseless_corrupted, num_sc_ambient, num_sc_adj

In [ ]:
# moons
%autoreload 2
n_points = 2500
noise_thresh = None

noise = 0.1
return_dict = moons(n_points=n_points, noise=noise, noise_thresh=noise_thresh)
noiseless_data = return_dict['noiseless_data']
G, G_noiseless_corrupted, num_sc_ambient, num_sc_adj = noise_model_experiment(noiseless_data, return_dict['cluster'], noise, dataset_name='moons', pmax=0.4, sigma_1_sq=0.11, sigma_2_sq=0.01, n_iter=50, n_noisy=1, k=15)


In [ ]:
n_points = 2500
# concentric circles
noise = 0.1
noise_thresh = None

return_dict = concentric_circles(n_points=n_points, factor=0.4, noise=noise, noise_thresh=noise_thresh)
noiseless_data = return_dict['noiseless_data']
G, G_noiseless_corrupted, num_sc_ambient, num_sc_adj = noise_model_experiment(noiseless_data, return_dict['cluster'], noise, dataset_name='circles', pmax=0.8, sigma_1_sq=0.11, sigma_2_sq=0.01, n_iter=50, n_noisy=1, k=15)

In [ ]:
# double swiss roll
noise = 0.125
return_dict = swiss_roll(n_points=3000, noise=noise, noise_thresh=None, double=True)
noiseless_data = return_dict['noiseless_data']
G, G_noiseless_corrupted, num_sc_ambient, num_sc_adj = noise_model_experiment(noiseless_data, return_dict['cluster'], noise, dataset_name='double_swiss_roll', pmax=0.175, sigma_1_sq=4, sigma_2_sq=0.125, n_iter=50, n_noisy=1, k=15)

In [ ]:
# double torus
noise = 0.5
return_dict = torus(n_points=3000, noise=0, noise_thresh=None, double=True)
noiseless_data = return_dict['noiseless_data']
G, G_noiseless_corrupted, num_sc_ambient, num_sc_adj = noise_model_experiment(noiseless_data, return_dict['cluster'], noise, dataset_name='double_torus', pmax=0.275, sigma_1_sq=5, sigma_2_sq=0.85, n_iter=50, n_noisy=15, k=15)